In [1]:
import torch
import gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

/home/damian/New Folder/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
wiki_plots = load_dataset("vishnupriyavr/wiki-movie-plots-with-summaries", split='train')
wiki_plots

Dataset({
    features: ['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot', 'PlotSummary'],
    num_rows: 34886
})

In [13]:
print(wiki_plots['Title'][0], ": :", wiki_plots['PlotSummary'][0])

Kansas Saloon Smashers : : Carrie Nation and her followers burst into a saloon and attack a bartender. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.


In [5]:
plot_summaries = [wiki_plot for wiki_plot in wiki_plots['PlotSummary']]
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
plot_encodings = sbert_model.encode(plot_summaries, convert_to_tensor=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4662.90it/s]


In [6]:
def find_similarities(description, top_k=3):
    description_encoding = sbert_model.encode([description], convert_to_tensor=True)
    similarities = sbert_model.similarity(description_encoding, plot_encodings)
    sim_indices = similarities.topk(top_k).indices
    return [wiki_plots['Title'][i] for i in sim_indices]

In [44]:
find_similarities('ancient roman empire')

[['Quo Vadis', 'Sign of the Pagan', 'The Fall of the Roman Empire']]

# #4

In [5]:
qwen_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct')
qwen_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-7B-Instruct', device_map='auto', dtype='auto')

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 129.12it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

class MovieExpert:
    def __init__(self, model, tokenizer, max_answer_length=10000):
        self.context = """You are Assistant. Assistant is a movie expert.

You have access to a movie database through RAG.

Whenever the user asks for a movie recommendation, you MUST request
information from the database using exactly this format:

<RAG_START>concise search query<RAG_END>

Example 1: \nUser: Hi, how are you? could you recommend me some scary movies about zombies?\nAssistant: Hi, I am great, thanks! what about you? Sure, there some movies I'd like to recommend you <RAG_START>scary movie about zombies<RAG_END>
Example 2: \nUser: recommend me a film about world war 2\nAssistant: Sure! Here are some films about world war 2: <RAG_START> movie world war 2 <RAG_END>
Example 3: \nUser: Hello! can you advice me something to watch about cars and races?\nAssistant: Hello!, Of course I can!\nHere are my recommendations: <RAG_START> cars and races <RAG_END>

Do not answer the recommendation until the database information has
been provided.

For all other questions, answer normally."""

        self.model = model
        self.tokenizer = tokenizer
        self.max_answer_length = max_answer_length
        self.rag_flag = 0
        
    def chat(self, prompt):
        self.context += "\nUser: " + prompt + "\nAssistant:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context, max_new_tokens=50)
            answer = extended[start_index:]
            
            if "<RAG_START>" in answer and "<RAG_END>" in answer:
                request = answer.split('<RAG_START>')[1].split('<RAG_END>')
                movies_lst = find_similarities(request)
                movies = "".join(i + " " for i in movies_lst)
                answer = answer.replace('<RAG_START>' + request + '<RAG_END>', movies)
                self.rag_flag += 1
            
            if ('\nUser:' in answer or extended==context or len(answer)>self.max_answer_length): break
            context = extended
        answer = answer.split('\nUser:')[0]
        self.context += answer
        return answer.strip()

In [2]:
gc.collect()
torch.cuda.empty_cache()

In [2]:
qwen_tokenizer_2 = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
qwen_model_2 = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', device_map='auto', dtype='auto')

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 991.36it/s]


In [4]:
few_shot_learning = ["\nUser: Hi, how are you? could you recommend me some scary movies about zombies?\nAssistant: Hi, I am great, thanks! what about you? Sure, there some movies I'd like to recommend you <RAG_START>scary movie about zombies<RAG_END>",
                     "\nUser: recommend me a film about world war 2\nAssistant: Sure! Here are some films about world war 2: <RAG_START> movie world war 2 <RAG_END>",
                     "\nUser: Hello! can you advice me something to watch about cars and races?\nAssistant: Hello!, Of course I can!\nHere are my recommendations: <RAG_START> cars and races <RAG_END>"]
few_shot_learning = Dataset.from_dict({'text' : few_shot_learning})

In [5]:
sft_output_dir = './sft_ex3'
sft_config = SFTConfig(sft_output_dir, per_device_train_batch_size=1, num_train_epochs=5)
sft_train = SFTTrainer(qwen_model_2, sft_config, processing_class=qwen_tokenizer_2, train_dataset=few_shot_learning)

Adding EOS to train dataset: 100%|██████████| 3/3 [00:00<00:00, 1452.66 examples/s]
Parameter 'function'=<function SFTTrainer._prepare_dataset.<locals>.tokenize_fn at 0x7fc8346ef3d0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Truncating train dataset: 100%|██████████| 3/3 [00:00<00:00, 1150.70 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 3/3 [00:00<00:00, 1468.25 examples/s]


In [6]:
train_output = sft_train.train()
sft_train.model.save_pretrained(sft_output_dir)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB. GPU 0 has a total capacity of 5.66 GiB of which 28.69 MiB is free. Including non-PyTorch memory, this process has 5.16 GiB memory in use. Of the allocated memory 4.99 GiB is allocated by PyTorch, and 57.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [14]:
test_movie_expert = MovieExpert(qwen_model_2, qwen_tokenizer_2)
test_movie_expert.chat('could you advice me a movie about Roman Empire?')

'Sure! Here\'s a concise search query I can use to find movies related to the Roman Empire:\n"movies about Roman Empire"'

In [15]:
test_movie_expert.rag_flag

0

In [11]:
test = "Hello, my name is Bob, and I am 20"
chunk = test.split('my')[1].split('Bob')[0]
print(test)
print(chunk)
test = test.replace('my' + chunk + 'Bob', 'I love cats')
print(test)
tttest = ["hello", 'I am', 'Bob']
print("".join(i + " " for i in tttest))

Hello, my name is Bob, and I am 20
 name is 
Hello, I love cats, and I am 20
hello I am Bob 
